# M4 · 의미 임베딩 색인과 검색

08에서 확정한 청크 전체를 OpenAI 임베딩으로 변환해 개인 Chroma에 별도 저장한다. 기존 문자 검색 색인은 유지한다. 다시 실행하면 이미 색인한 청크는 재호출하지 않는다.

In [ ]:
from pathlib import Path
import hashlib,json
from IPython.display import display
from src.semantic_index import build
from src.retriever import PersonalRetriever
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
CORPUS=ROOT/'data'/'personal'/'corpus'

## 전체 청크 색인과 원본 일치 확인

In [ ]:
meta=build(ROOT)
assert meta['complete'] and meta['indexed']==meta['chunks']==6845
lexical=json.loads((CORPUS/'index_meta.json').read_text(encoding='utf-8'))
assert lexical['corpus_sha256']==meta['corpus_sha256']
assert hashlib.sha256((CORPUS/'chunks.jsonl').read_bytes()).hexdigest()==meta['corpus_sha256']
display({k:meta[k] for k in ('collection','chunks','indexed','model','dimensions','complete')})

## 의미 검색의 실제 문서 확인

In [ ]:
retriever=PersonalRetriever(ROOT,mode='semantic')
for label,question,kind in [('지침','이동식 사다리 작업 전에 확인할 사항','kosha_guide'),('사례','드럼통 안 화학물질이 산소절단 중 폭발한 사고','sif')]:
    hits=retriever.retrieve(question,k=5,source_type=kind)
    print(label,question)
    display([{'doc_id':h['doc_id'],'제목':h['metadata']['title'],'쪽':h['metadata']['page'],'유사도':round(h['similarity'],3)} for h in hits])

## 의미 색인 결과와 비용·사용 주의

- 6,845개 청크가 `text-embedding-3-small` 의미 임베딩 1,536차원으로 변환돼 Chroma 색인에 들어갔다. 색인 완료 수, 청크 수, 원본 코퍼스 SHA-256이 일치하는 것을 기록했다.
- 질의도 같은 모델로 임베딩해야 검색할 수 있어 이후 질문마다 임베딩 API 비용과 호출 제한이 발생한다. 문서 재적재 때 중복 호출을 줄이도록 기존 동일 코퍼스 색인을 재사용한다.
- 이 결과는 검색 가능한 색인이 완성됐다는 확인이지 관련 질문에 항상 맞는 근거를 찾는다는 보증은 아니다. Hit@K·MRR 평가는 M6에서 별도 수행한다.
- ⑥·⑧ PDF의 OCR 문장은 인식 오류가 있을 수 있어 근거 페이지를 열어 확인한다. 빈 페이지 6쪽은 빠졌으므로 해당 내용이 검색되지 않을 수 있다.

### 다음에 이어서

- M5에서 검색 결과와 질문을 LLM에 전달하고 출처 인용·모름 처리 규칙을 검사한다.
- 코퍼스 수정 시 기존 임베딩과 새 코퍼스를 섞지 말고 해시·청크 수를 다시 확인한다.
- 사용량·비용 검토 파일 `data/personal/evaluation/embedding_cost_review.md`를 재적재 전에 확인한다.
